# Advanced C# for .NET Core Architects

This notebook goes deeper into C# building blocks that are heavily used in modern .NET Core apps:
- Delegates and events
- Interfaces vs abstract classes (for DI and architecture)
- `IEnumerable<T>` vs `IQueryable<T>`
- Attributes and reflection (annotations)

Run each code cell, then modify it to see how behavior changes.


## 1. Delegates and Events

Delegates are types that represent methods. Events build on delegates to implement the observer pattern.
In real apps, events are used for domain events, UI events, and background notifications.


In [ ]:
// A delegate type: any method that matches (string message) can be attached.
delegate void NotificationHandler(string message);

class Notifier
{
    public event NotificationHandler? OnNotify;

    public void DoWork()
    {
        // ... some work
        OnNotify?.Invoke("Work completed.");
    }
}

void Logger(string message) => Console.WriteLine($"LOG: {message}");

var notifier = new Notifier();
notifier.OnNotify += Logger;
notifier.DoWork();


## 2. Interfaces vs Abstract Classes (for DI)

Interfaces define *contracts*; abstract classes can provide a partial default implementation.
In ASP.NET Core you typically inject interfaces into constructors to keep code testable and flexible.


In [ ]:
interface IEmailSender
{
    void Send(string to, string subject, string body);
}

class SmtpEmailSender : IEmailSender
{
    public void Send(string to, string subject, string body)
    {
        Console.WriteLine($"Sending email to {to}: {subject}");
    }
}

abstract class BaseController
{
    protected readonly IEmailSender EmailSender;
    protected BaseController(IEmailSender emailSender) => EmailSender = emailSender;
}

class UserController : BaseController
{
    public UserController(IEmailSender sender) : base(sender) { }

    public void Register(string email)
    {
        // domain logic ...
        EmailSender.Send(email, "Welcome", "Thanks for registering.");
    }
}

IEmailSender emailSender = new SmtpEmailSender();
var controller = new UserController(emailSender);
controller.Register("user@example.com");


## 3. `IEnumerable<T>` vs `IQueryable<T>`

- `IEnumerable<T>`: in-memory collections, LINQ runs in .NET (good for lists, arrays).
- `IQueryable<T>`: query is *translated* (e.g., to SQL in EF Core) and executed by an external provider.

Key idea: with `IQueryable<T>` **defer materialization** and let the database do the heavy lifting.


In [ ]:
using System.Collections.Generic;
using System.Linq;

List<int> numbers = new() { 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 };
IEnumerable<int> evenEnumerable = numbers.Where(n => n % 2 == 0);
Console.WriteLine("IEnumerable result: " + string.Join(", ", evenEnumerable));

IQueryable<int> queryable = numbers.AsQueryable();
IQueryable<int> evenQueryable = queryable.Where(n => n % 2 == 0);
Console.WriteLine("IQueryable provider: " + evenQueryable.Provider.GetType().Name);
Console.WriteLine("IQueryable expression: " + evenQueryable.Expression);
Console.WriteLine("IQueryable materialized: " + string.Join(", ", evenQueryable.ToList()));


## 4. Attributes and Reflection

Attributes are metadata you attach to types and members (similar to annotations).
Frameworks (ASP.NET Core, EF Core, testing frameworks) read attributes via reflection to change behavior.


In [ ]:
using System;
using System.Reflection;

[AttributeUsage(AttributeTargets.Class | AttributeTargets.Method)]
class AuditAttribute : Attribute
{
    public string Action { get; }
    public AuditAttribute(string action) => Action = action;
}

[Audit("CreateOrder")]
class OrderService
{
    [Audit("PlaceOrder")]
    public void PlaceOrder() { }
}

var type = typeof(OrderService);
var classAudit = type.GetCustomAttribute<AuditAttribute>();
Console.WriteLine($"Class Audit Action: {classAudit?.Action}");

var method = type.GetMethod("PlaceOrder");
var methodAudit = method?.GetCustomAttribute<AuditAttribute>();
Console.WriteLine($"Method Audit Action: {methodAudit?.Action}");
